In [ ]:
# Cell 0 — Install dependencies
!pip install unsloth
!pip install -q git+https://github.com/feralvam/easse.git
!pip install -q sacrebleu
print("Done — restarting now...")
import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 1 — Imports
import unsloth
from unsloth import FastLanguageModel
import torch, json, re, time
from collections import defaultdict
print("✅ Imports done")

In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'
print("✅ Drive mounted")

In [ ]:
# Cell 3 — load_jsonl and simplify_text
import json, torch

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

INSTRUCTION = (
    'Tu es un expert en orthophonie et en éducation inclusive. '
    'Simplifie le texte suivant pour un enfant dyslexique francophone âgé de 6 à 8 ans. '
    'Applique ces règles obligatoires : '
    '(1) Phrases courtes de 8 à 10 mots maximum. '
    '(2) Structure Sujet + Verbe + Complément uniquement. '
    '(3) Vocabulaire simple — remplace les mots de plus de 3 syllabes. '
    '(4) Découpage syllabique avec tirets : ma-man, jar-din, é-lè-ve. '
    '(5) Conserve TOUS les détails narratifs. '
    '(6) Retour à la ligne à chaque phrase.'
)

def simplify_text(model, tokenizer, text, max_new_tokens=600):
    prompt = f'### Instruction:\n{INSTRUCTION}\n\n### Input:\n{text}\n\n### Response:\n'
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=1024
    ).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True)
    if '### ' in result:
        result = result[:result.index('### ')].strip()
    return result

print('✅ Functions defined')

In [ ]:
# Cell 4 — Zero-shot vs Model 3 output response comparison (for 3 instances: T174 and T316 and T054)
import gc, torch
from unsloth import FastLanguageModel

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

# Texts to compare
examples = [
    {
        'id': 'T174',
        'text': "Un chat est sur le toit. Il regarde les oiseaux. Un oiseau s'envole. Le chat saute. Il manque l'oiseau."
    },
    {
        'id': 'T316',
        'text': "C'est le soir, Lila a rendez-vous avec ses amis imaginaires. Elle voit un monstre qui s'appelle Doli, qui a un œil et une grande bouche. Ce monstre a deux cornes et deux pattes. Il se cache sous les lits. Il aime manger des bananes et des bonbons."
    },
    {
        'id': 'T054',
        'text': "En 1960, mon grand-père avait vingt ans. Sa première voiture était une 2 CV. On l'appelait aussi la « Deudeuche » ou la « deux pattes ». Il nous en parle souvent. Elle représente toute une époque. C'était une voiture française qui ne coûtait pas cher et qui était facile à réparer. Elle passait partout, en ville comme dans les champs."
    },
]

# ── Zero-shot (base Mistral, no fine-tuning) ───────────────────────
print("Loading zero-shot model...")
model_zs, tokenizer_zs = FastLanguageModel.from_pretrained(
    model_name='unsloth/mistral-7b-instruct-v0.2-bnb-4bit',
    max_seq_length=1024, dtype=None, load_in_4bit=True,
)
model_zs = FastLanguageModel.for_inference(model_zs)
tokenizer_zs.pad_token = tokenizer_zs.eos_token

zeroshot_outputs = {}
for ex in examples:
    zs = simplify_text(model_zs, tokenizer_zs, ex['text'])
    zeroshot_outputs[ex['id']] = zs
    print(f"\n{'='*55}")
    print(f"ZERO-SHOT — {ex['id']}")
    print(f"ORIGINAL:  {ex['text']}")
    print(f"OUTPUT:    {zs}")

del model_zs, tokenizer_zs
gc.collect()
torch.cuda.empty_cache()

# ── Model 3 (fine-tuned) ───────────────────────────────────────────
print("\nLoading Model 3...")
model3, tokenizer3 = FastLanguageModel.from_pretrained(
    model_name=DRIVE_PATH + 'model_v3_retrained/',
    max_seq_length=1024, dtype=None, load_in_4bit=True,
)
model3 = FastLanguageModel.for_inference(model3)
tokenizer3.pad_token = tokenizer3.eos_token

model3_outputs = {}
for ex in examples:
    m3 = simplify_text(model3, tokenizer3, ex['text'])
    model3_outputs[ex['id']] = m3
    print(f"\n{'='*55}")
    print(f"MODEL 3 — {ex['id']}")
    print(f"ORIGINAL:  {ex['text']}")
    print(f"OUTPUT:    {m3}")

del model3, tokenizer3
gc.collect()
torch.cuda.empty_cache()

# ── Summary ────────────────────────────────────────────────────────
print("\n" + "="*55)
print("SUMMARY COMPARISON")
print("="*55)
for ex in examples:
    print(f"\n--- {ex['id']} ---")
    print(f"ORIGINAL:   {ex['text']}")
    print(f"ZERO-SHOT:  {zeroshot_outputs[ex['id']]}")
    print(f"MODEL 3:    {model3_outputs[ex['id']]}")

# Save
results = [
    {
        'id': ex['id'],
        'original': ex['text'],
        'zeroshot': zeroshot_outputs[ex['id']],
        'model3': model3_outputs[ex['id']],
    }
    for ex in examples
]
with open(DRIVE_PATH + 'zeroshot_vs_model3.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('\n✅ Results saved to zeroshot_vs_model3.json')

In [ ]:
# Cell 5 — SARI component breakdown (Model 3)
import json
from easse.sari import get_corpus_sari_operation_scores

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

outputs3 = json.load(open(DRIVE_PATH + 'outputs_model3.json', encoding='utf-8'))

sources     = [ex['original'] for ex in outputs3]
predictions = [ex['model_output'] for ex in outputs3]

# ← correct shape: list of 1 reference list containing all 36 refs
references  = [[ex['human_simplified'] for ex in outputs3]]

add, keep, delete = get_corpus_sari_operation_scores(
    orig_sents=sources,
    sys_sents=predictions,
    refs_sents=references,
)

print(f'SARI Add:    {add:.2f}')
print(f'SARI Keep:   {keep:.2f}')
print(f'SARI Delete: {delete:.2f}')
print(f'SARI Total:  {(add+keep+delete)/3:.2f}')

In [ ]:
# Cell 6 — Inference time measurement (Model 3)
import time, json, gc, torch
from unsloth import FastLanguageModel

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'
test_data = load_jsonl(DRIVE_PATH + 'dataset_test_fixed.jsonl')

model3, tokenizer3 = FastLanguageModel.from_pretrained(
    model_name=DRIVE_PATH + 'model_v3_retrained/',
    max_seq_length=1024, dtype=None, load_in_4bit=True,
)
model3 = FastLanguageModel.for_inference(model3)
tokenizer3.pad_token = tokenizer3.eos_token

# Time 10 inferences for reliable average
times = []
for ex in test_data[:10]:
    t0 = time.time()
    _ = simplify_text(model3, tokenizer3, ex['input'])
    times.append(time.time() - t0)

print(f'Inference time over 10 texts:')
print(f'  Average : {sum(times)/len(times):.2f} seconds')
print(f'  Min     : {min(times):.2f} seconds')
print(f'  Max     : {max(times):.2f} seconds')

del model3, tokenizer3
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 7 — Error analysis by niveau (CP vs CE1)
import json, re
from collections import defaultdict

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'
outputs3 = json.load(open(DRIVE_PATH + 'outputs_model3.json', encoding='utf-8'))

def map_niveau(n):
    n = n.lower()
    if 'cp' in n: return 'CP'
    if 'ce1' in n: return 'CE1'
    return 'Other'

def avg_sent_len(text):
    text = re.sub(r'(?<=[a-zA-ZÀ-ÿ])-(?=[a-zA-ZÀ-ÿ])', '', text)
    sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    words = re.findall(r"[a-zA-ZÀ-ÿ']+", text.lower())
    return len(words)/len(sents) if sents else 0

def count_words(text):
    text = re.sub(r'(?<=[a-zA-ZÀ-ÿ])-(?=[a-zA-ZÀ-ÿ])', '', text)
    return len(re.findall(r"[a-zA-ZÀ-ÿ']+", text.lower()))

by_niveau = defaultdict(lambda: {'orig_asl':[], 'model_asl':[], 'orig_words':[], 'model_words':[], 'n':0})

for ex in outputs3:
    niv = map_niveau(ex['niveau'])
    by_niveau[niv]['orig_asl'].append(avg_sent_len(ex['original']))
    by_niveau[niv]['model_asl'].append(avg_sent_len(ex['model_output']))
    by_niveau[niv]['orig_words'].append(count_words(ex['original']))
    by_niveau[niv]['model_words'].append(count_words(ex['model_output']))
    by_niveau[niv]['n'] += 1

print(f"{'Niveau':<8} {'N':>4} {'Orig ASL':>10} {'Model ASL':>10} {'Δ ASL':>8} {'Orig Words':>12} {'Model Words':>12} {'Δ Words':>8}")
print('-'*80)
for niv in ['CP', 'CE1']:
    d = by_niveau[niv]
    n = d['n']
    o_asl = sum(d['orig_asl'])/n
    m_asl = sum(d['model_asl'])/n
    o_w   = sum(d['orig_words'])/n
    m_w   = sum(d['model_words'])/n
    print(f"{niv:<8} {n:>4} {o_asl:>10.2f} {m_asl:>10.2f} {m_asl-o_asl:>+8.2f} {o_w:>12.1f} {m_w:>12.1f} {m_w-o_w:>+8.1f}")

In [ ]:
# Cell 8 — Pyphen post-processing syllabifier
!pip install pyphen -q

import pyphen
import re
import json

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

# Load Model 3 outputs
outputs3 = json.load(open(DRIVE_PATH + 'outputs_model3.json', encoding='utf-8'))

# Initialize French syllabifier
dic = pyphen.Pyphen(lang='fr_FR')

def syllabify_word(word):
    """Syllabify a single French word using pyphen."""
    # Strip punctuation for syllabification then reattach
    clean = re.sub(r"[^a-zA-ZÀ-ÿ]", "", word)
    if len(clean) <= 2:
        return word  # don't hyphenate very short words
    syllabified = dic.inserted(clean, hyphen='-')
    return syllabified if syllabified else word

def post_process_syllabification(text):
    """
    Replace existing syllabic segmentation in model output
    with pyphen-verified segmentation.
    """
    lines = text.split('\n')
    result_lines = []
    for line in lines:
        # Split line into tokens (words + punctuation)
        tokens = re.findall(r"[a-zA-ZÀ-ÿ'-]+|[^a-zA-ZÀ-ÿ'-]+", line)
        new_tokens = []
        for token in tokens:
            # Check if token already has syllabic hyphens
            if '-' in token and re.match(r"[a-zA-ZÀ-ÿ]+-[a-zA-ZÀ-ÿ]", token):
                # Remove existing hyphens and re-syllabify with pyphen
                word = token.replace('-', '')
                new_tokens.append(syllabify_word(word))
            elif re.match(r"^[a-zA-ZÀ-ÿ]+$", token) and len(token) > 2:
                # Unsyllabified word — syllabify it
                new_tokens.append(syllabify_word(token))
            else:
                new_tokens.append(token)
        result_lines.append(''.join(new_tokens))
    return '\n'.join(result_lines)

# Apply to all outputs
outputs3_pyphen = []
for ex in outputs3:
    new_output = post_process_syllabification(ex['model_output'])
    outputs3_pyphen.append({
        'id':               ex['id'],
        'niveau':           ex['niveau'],
        'original':         ex['original'],
        'human_simplified': ex['human_simplified'],
        'model_output':     ex['model_output'],
        'model_output_pyphen': new_output,
    })

# Save
with open(DRIVE_PATH + 'outputs_model3_pyphen.json', 'w', encoding='utf-8') as f:
    json.dump(outputs3_pyphen, f, ensure_ascii=False, indent=2)

# Show comparison for key examples
for target_id in ['T091', 'T174', 'T231', 'T316']:
    ex = next((e for e in outputs3_pyphen if e['id'] == target_id), None)
    if ex:
        print(f"\n{'='*55}")
        print(f"ID: {target_id}")
        print(f"BEFORE: {ex['model_output'][:200]}")
        print(f"AFTER:  {ex['model_output_pyphen'][:200]}")

print('\n✅ Saved to outputs_model3_pyphen.json')